In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from faker import Faker

fake = Faker('fr_FR')  # Données françaises (adaptez si besoin)
random.seed(42)
np.random.seed(42)

# Paramètres
nb_users = 500
nb_accounts = 800  # certains users ont plusieurs comptes
nb_transactions = 15000
start_date = datetime(2024, 1, 1)
end_date = datetime(2025, 3, 31)

# ---------------------------
# 1. Table users
# ---------------------------
users = []
for i in range(1, nb_users + 1):
    users.append({
        'user_id': i,
        'full_name': fake.name(),
        'phone_number': fake.unique.phone_number(),
        'email': fake.email(),
        'registration_date': fake.date_between(start_date=start_date, end_date=end_date),
        'status': random.choices(['active', 'inactive', 'suspended'], weights=[0.85, 0.10, 0.05])[0],
        'kyc_level': random.choices([1, 2, 3], weights=[0.2, 0.5, 0.3])[0],
        'city': fake.city(),
        'date_of_birth': fake.date_of_birth(minimum_age=18, maximum_age=75)
    })
df_users = pd.DataFrame(users)

# ---------------------------
# 2. Table accounts (plusieurs comptes par user)
# ---------------------------
accounts = []
account_types = ['principal', 'savings', 'merchant', 'salary']
for acc_id in range(1, nb_accounts + 1):
    user_id = random.randint(1, nb_users)
    # Éviter qu'un même user ait deux comptes de même type (optionnel)
    accounts.append({
        'account_id': acc_id,
        'user_id': user_id,
        'account_type': random.choice(account_types),
        'currency': 'XOF',  # Franc CFA
        'balance': round(random.uniform(0, 500000), 2),  # solde initial
        'opening_date': fake.date_between(start_date=start_date, end_date=end_date),
        'is_active': random.choices([True, False], weights=[0.9, 0.1])[0]
    })
df_accounts = pd.DataFrame(accounts)

# ---------------------------
# 3. Table transactions (comportements variés)
# ---------------------------
transactions = []
transaction_types = ['transfer', 'cash_in', 'cash_out', 'payment', 'airtime_purchase']
statuses = ['completed', 'pending', 'failed', 'reversed']

# Pour générer des comportements anormaux
def random_amount(trans_type):
    if trans_type == 'cash_in':
        return round(random.uniform(100, 500000), 2)
    elif trans_type == 'cash_out':
        return round(random.uniform(100, 200000), 2)
    elif trans_type == 'payment':
        return round(random.uniform(50, 50000), 2)
    else:  # transfer, airtime
        return round(random.uniform(10, 100000), 2)

for t_id in range(1, nb_transactions + 1):
    # Choisir deux comptes (source et destination)
    source_account = random.choice(df_accounts['account_id'].tolist())
    dest_account = random.choice(df_accounts['account_id'].tolist())
    # Éviter auto-transaction dans 95% des cas
    while source_account == dest_account and random.random() < 0.95:
        dest_account = random.choice(df_accounts['account_id'].tolist())
    
    trans_type = random.choice(transaction_types)
    amount = random_amount(trans_type)
    
    # Ajouter des anomalies : montant nul, montant négatif, date future, etc.
    anomaly_prob = 0.03  # 3% de transactions anormales
    if random.random() < anomaly_prob:
        # anomalie : montant négatif
        amount = -amount
    elif random.random() < 0.01:
        # anomalie : montant zéro
        amount = 0.0
    
    # Date aléatoire entre start_date et end_date, avec plus de transactions récentes
    date_offset = random.expovariate(1/30)  # plus de transactions récentes
    trans_date = end_date - timedelta(days=min(date_offset, (end_date-start_date).days))
    if trans_date < start_date:
        trans_date = start_date
    
    status = random.choices(statuses, weights=[0.85, 0.05, 0.07, 0.03])[0]
    
    # Pour certaines transactions échouées, on met un motif
    failure_reason = None
    if status in ['failed', 'reversed']:
        failure_reason = random.choice(['insufficient_balance', 'technical_error', 'limit_exceeded', 'suspicious'])
    
    transactions.append({
        'transaction_id': t_id,
        'account_source_id': source_account,
        'account_dest_id': dest_account,
        'transaction_type': trans_type,
        'amount': amount,
        'transaction_date': trans_date.strftime('%Y-%m-%d %H:%M:%S'),
        'status': status,
        'failure_reason': failure_reason,
        'device_id': fake.uuid4(),
        'location': fake.city()
    })
df_transactions = pd.DataFrame(transactions)

# ---------------------------
# 4. Table fraud_alerts (liée aux transactions suspectes)
# ---------------------------
fraud_alerts = []
alert_types = ['unusual_amount', 'rapid_succession', 'location_mismatch', 'multiple_failures']

# On va marquer certaines transactions comme frauduleuses
# Transactions suspectes : montant > 200000 ou échecs répétés ou montant négatif
suspect_trans_ids = df_transactions[
    (df_transactions['amount'] > 200000) |
    (df_transactions['amount'] < 0) |
    (df_transactions['status'].isin(['failed', 'reversed']))
]['transaction_id'].tolist()

for idx, trans_id in enumerate(suspect_trans_ids[:int(len(suspect_trans_ids)*0.7)], start=1):
    fraud_alerts.append({
        'alert_id': idx,
        'transaction_id': trans_id,
        'alert_type': random.choice(alert_types),
        'alert_date': fake.date_time_between(start_date=start_date, end_date=end_date),
        'severity': random.choice(['low', 'medium', 'high']),
        'resolved': random.choice([True, False])
    })
df_fraud = pd.DataFrame(fraud_alerts)

# ---------------------------
# 5. Table daily_balance_snapshot (soldes journaliers des comptes)
# ---------------------------
# Pour chaque compte, on génère un solde à chaque jour (ou presque) depuis son ouverture jusqu'à end_date
snapshots = []
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

for acc_id in df_accounts['account_id'].unique():
    # Récupérer la date d'ouverture du compte
    opening = pd.to_datetime(df_accounts[df_accounts['account_id']==acc_id]['opening_date'].values[0])
    # Simuler une série de soldes avec marche aléatoire
    balance = df_accounts[df_accounts['account_id']==acc_id]['balance'].values[0]
    for day in date_range:
        if day < opening:
            continue
        # Variation aléatoire du solde entre -5000 et +8000, mais pas négatif
        change = np.random.normal(0, 2000)
        balance = max(0, balance + change)
        # Ajouter un peu de bruit
        snapshots.append({
            'snapshot_id': len(snapshots)+1,
            'account_id': acc_id,
            'snapshot_date': day.strftime('%Y-%m-%d'),
            'closing_balance': round(balance, 2)
        })
        # On ne prend qu'un snapshot par jour, déjà fait
df_snapshots = pd.DataFrame(snapshots)
# Limiter à 50000 lignes max pour rester raisonnable
if len(df_snapshots) > 50000:
    df_snapshots = df_snapshots.sample(50000, random_state=42)

# ---------------------------
# Sauvegarde en CSV
# ---------------------------
df_users.to_csv('fintech_users.csv', index=False)
df_accounts.to_csv('fintech_accounts.csv', index=False)
df_transactions.to_csv('fintech_transactions.csv', index=False)
df_fraud.to_csv('fintech_fraud_alerts.csv', index=False)
df_snapshots.to_csv('fintech_daily_balance.csv', index=False)

print("5 fichiers CSV générés avec succès !")
print(f" - users: {len(df_users)} lignes")
print(f" - accounts: {len(df_accounts)} lignes")
print(f" - transactions: {len(df_transactions)} lignes")
print(f" - fraud_alerts: {len(df_fraud)} lignes")
print(f" - daily_balance: {len(df_snapshots)} lignes")

5 fichiers CSV générés avec succès !
 - users: 500 lignes
 - accounts: 800 lignes
 - transactions: 15000 lignes
 - fraud_alerts: 2395 lignes
 - daily_balance: 50000 lignes
